# 24 · Should batters be compared only with batters in the same position?

> **Run on the first data file.** This research notebook and its saved outputs come from the ball-by-ball file as first supplied (every extra counted as 1 run). On 16 Nov 2026 the organisers issued a corrected file with the real extra runs; the method was re-run and re-checked on it in [notebook 28](28_final_review_corrected_data.ipynb), and both top-10 lists kept the same names.

Bowlers are compared pace with pace and spin with spin. Would the same idea help batters? Here each batter is compared only with batters in the same batting position (openers, No. 3-4, No. 5-6, No. 7+), and then also within the same phase of the innings. Judged on the same 52 internal checks as notebook 11, with the rule fixed in advance.

**Result.** No position gets an unfair boost: the averages of all four positions are close to zero. Both versions pick **the same 10 batters** and predict hidden matches slightly worse (0.137 against 0.143), so the current rating stays.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes and the rule fixed before running</summary>

P27 - Should batters be compared only with batters in the same batting position? (research; nothing existing is changed)  
  
User's idea (2026-11-15): bowlers are compared pace v pace and spin v spin; do the same for batters.  
The fixed 'type' of a batter is his batting position (the job), not his style (anchor / accelerator is defined by the  
performance being measured, so a style baseline would be circular).  
Diagnosis (run before this script): average skill score per 100 weighted balls by position: openers +0.3, No. 3-4 +1.0,  
No. 5-6 +0.6, No. 7+ -1.3 (each +/- 0.4 to 0.8); top 30 mirrors the pool; top 10 has 5 openers v about 3.6 expected.  
Rule fixed BEFORE running:  
  A  position-relative: each ball's skill score minus the average for its batting position (1-2, 3-4, 5-6, 7+),  
     computed on the training matches of each split only;  
  B  position x phase relative: the same within each position and phase (powerplay, middle, death).  
  Judged on the same 52 internal checks as P9 (and 2026 for information). A version replaces the current rating only if it  
  is at least as good (objective within one paired standard error) AND it removes a real tilt (top-10 position mix closer  
  to the pool). Runs sequentially on 2 threads.
</details>

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from rrlib.common import CFG, SEED, load, load_json, load_table, save_table, save_json, get_logger, Timer, weighted_spearman, excluded_reasons
from rrlib import ratings as RL
try:
    import psutil
    psutil.Process().nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass

log = get_logger("p27_batting_position_baseline")
v = load("values")
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]["bat"]
slot = pd.cut(v.batting_order, [0, 2, 4, 6, 11], labels=False).fillna(3).astype(int).values
v["slot"] = slot
v["w_"] = v.pressure_pct.clip(1e-3, 1) ** 2


def add_relative(df, fit_on):
    """returns df with v9_bat (position-relative) and v10_bat (position x phase relative), offsets from fit_on rows"""
    f = fit_on[fit_on.faced.astype(bool) & fit_on.v3_bat.notna()]
    o1 = f.assign(wv=f.w_ * f.v3_bat).groupby("slot")[["wv", "w_"]].sum()
    o1 = (o1.wv / o1.w_)
    o2 = f.assign(wv=f.w_ * f.v3_bat).groupby(["slot", "phase"])[["wv", "w_"]].sum()
    o2 = (o2.wv / o2.w_)
    out = df.copy()
    out["v9_bat"] = out.v3_bat - out.slot.map(o1).values
    key = pd.MultiIndex.from_arrays([out.slot.values, out.phase.values])
    out["v10_bat"] = out.v3_bat - o2.reindex(key).fillna(0).values
    return out


HOLD = CFG["cv"]["holdout_season"]
V = v[v.year < HOLD]
splits = [("rolling", f"{a[0]}-{a[-1]}->{b_}", V.year.isin(a).values, (V.year == b_).values) for a, b_ in CFG["cv"]["rolling"]]
mids = np.sort(V.match_id.unique()); rng = np.random.RandomState(SEED)
for i in range(CFG["cv"]["random_splits"]):
    a = set(rng.choice(mids, len(mids) // 2, replace=False)); m = V.match_id.isin(a).values
    splits.append(("random", f"r{i}", m, ~m))
splits.append(("holdout", "2022-25->2026", (v.year < HOLD).values, (v.year == HOLD).values))
VERS = {"v3": "current (vs average batter in the same situation)", "v9": "A: same batting position",
        "v10": "B: same batting position and phase"}
rows = []
with Timer(log, "52 checks + 2026, sequential"):
    for kind, name, tr_m, te_m in splits:
        base = v if kind == "holdout" else V
        tr = add_relative(base[tr_m], base[tr_m])
        te = base[te_m]
        t = te[te.is_pressure & te.v2_bat.notna()]
        yy = t.groupby("striker_id").agg(y2=("v2_bat", "mean"), y1=("v1_bat", "mean"), n=("v2_bat", "size")); yy = yy[yy.n >= 20]
        pool = yy.index[yy.index.isin(set(tr.loc[tr.is_pressure & tr.v1_bat.notna(), "striker_id"]))]
        if len(pool) < 15:
            continue
        for met in VERS:
            sc = RL.build_scores(tr, "bat", dict(chosen, value=met), comp).score.reindex(pool).fillna(-1e9)
            top = sc.sort_values(ascending=False).index[:10]
            wm = lambda col, idx: np.average(yy.loc[idx, col], weights=yy.loc[idx, "n"])
            rows.append({"kind": kind, "split": name, "version": met,
                         "rho": weighted_spearman(sc.values, yy.loc[pool].y2.values, yy.loc[pool].n.values),
                         "top10_lift_runs": 100 * (wm("y1", top) - wm("y1", pool))})
CV = pd.DataFrame(rows)
save_table(CV, "research_position_baseline_cv")
summ = {}
for met in VERS:
    r = CV[(CV.version == met) & (CV.kind != "holdout")]
    summ[met] = {"objective": float(0.5 * r[r.kind == "random"].rho.mean() + 0.5 * r[r.kind == "rolling"].rho.mean()),
                 "rolling": float(r[r.kind == "rolling"].rho.mean()), "top10_lift_runs": float(r.top10_lift_runs.mean()),
                 "holdout_2026": float(CV[(CV.version == met) & (CV.kind == "holdout")].rho.iloc[0])}
piv = CV[CV.kind == "random"].pivot(index="split", columns="version", values="rho")
for met in ("v9", "v10"):
    dd = piv[met] - piv["v3"]
    summ[met].update({"vs_current_diff": float(dd.mean()), "vs_current_se": float(dd.std(ddof=1) / np.sqrt(len(dd))), "better_splits": int((dd > 0).sum())})
# full-data lists and their position mix
full = add_relative(v, v)
EX = set(excluded_reasons()); names = load("players").set_index("player_id").name
b = v[v.faced.astype(bool)]
main_slot = b.groupby("striker_id").slot.agg(lambda s: s.value_counts().idxmax())
LAB = {0: "openers", 1: "No. 3-4", 2: "No. 5-6", 3: "No. 7+"}
submitted = load_table("final_top30_bat").head(10).name.tolist()
for met in VERS:
    sc = RL.build_scores(full, "bat", dict(chosen, value=met), comp)
    sc = sc[~sc.index.isin(EX)].sort_values("score", ascending=False)
    top = sc.index[:10]
    summ[met]["top10"] = [names.get(p) for p in top]
    summ[met]["same_names_as_submitted"] = len(set(summ[met]["top10"]) & set(submitted))
    summ[met]["position_mix_top10"] = {LAB[k]: int(c) for k, c in main_slot.reindex(top).value_counts().items()}
best = max(("v9", "v10"), key=lambda m_: summ[m_]["objective"])
ok = summ[best]["objective"] >= summ["v3"]["objective"] - summ[best]["vs_current_se"]
summ["decision"] = (f"position-relative version {best} is as good: consider it" if ok and summ[best]["same_names_as_submitted"] < 10
                    else "keep the current rating")
save_json(summ, "research_position_baseline_summary")
log.info("\n" + pd.DataFrame({k: {kk: vv for kk, vv in d.items() if kk not in ("top10", "position_mix_top10")}
                              for k, d in summ.items() if isinstance(d, dict)}).round(4).to_string())
for met, lab in VERS.items():
    log.info(f"{lab}: {summ[met]['position_mix_top10']} | {summ[met]['top10']}")
log.info(f"decision: {summ['decision']}")
